In [12]:
import torch
import torch.nn as nn
from torchvision.models import efficientnet_b0, EfficientNet_B0_Weights

# Same number of classes as training
num_classes = 41  

# Rebuild EfficientNet
model = efficientnet_b0(weights=None)  
model.classifier[1] = nn.Linear(model.classifier[1].in_features, num_classes)

# Load trained weights
model.load_state_dict(torch.load("efficientnet_70.pth", map_location="cpu"))
model.eval()

EfficientNet(
  (features): Sequential(
    (0): Conv2dNormActivation(
      (0): Conv2d(3, 32, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1), bias=False)
      (1): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (2): SiLU(inplace=True)
    )
    (1): Sequential(
      (0): MBConv(
        (block): Sequential(
          (0): Conv2dNormActivation(
            (0): Conv2d(32, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), groups=32, bias=False)
            (1): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
            (2): SiLU(inplace=True)
          )
          (1): SqueezeExcitation(
            (avgpool): AdaptiveAvgPool2d(output_size=1)
            (fc1): Conv2d(32, 8, kernel_size=(1, 1), stride=(1, 1))
            (fc2): Conv2d(8, 32, kernel_size=(1, 1), stride=(1, 1))
            (activation): SiLU(inplace=True)
            (scale_activation): Sigmoid()
          )
          (2): Conv2dNormActivat

In [13]:
from torchvision import transforms
from PIL import Image

# Same normalization as training
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize([0.5, 0.5, 0.5], [0.5, 0.5, 0.5])
])

# Load image
image_path = r"D:\model\images\Amritmahal.jpg"  # change to your image path
img = Image.open(image_path).convert("RGB")
input_tensor = transform(img).unsqueeze(0)  

In [14]:
import os
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)
input_tensor = input_tensor.to(device)

# Inference
with torch.no_grad():
    outputs = model(input_tensor)
    _, predicted = torch.max(outputs, 1)

# Map index to class name
# class_names = sorted([d for d in os.listdir("D:/Dataset/cow_crops_by_breed") 
                    #   if os.path.isdir(os.path.join("D:/Dataset/cow_crops_by_breed", d))])

print(f"Predicted Class Index: {predicted.item()}")
# print(f"Predicted Breed: {class_names[predicted.item()]}")

Predicted Class Index: 12


In [15]:
pip install ultralytics


Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.2 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [16]:
import torch
from torchvision import transforms, models
from PIL import Image
from ultralytics import YOLO
import os

# -----------------------------
# Settings
# -----------------------------
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
num_classes = 41  # number of cattle breeds

# -----------------------------
# KNOWN BREEDS MAPPING
# -----------------------------
class_names = {
    0: "Unknown_0", 
    1: "Amritmahal",          
    2: "Unknown_2", 
    3: "Unknown_3", 
    4: "Bargur",              
    5: "Banni",               
    6: "Purnea",              
    7: "Dangi",               
    8: "Binjharpuri",         
    9: "Gir",                 
    10: "Unknown_10", 
    11: "Shweta Kapila",      
    12: "Ghumusari",          
    13: "Holstein",           
    14: "Unknown_14", 
    15: "Lakhimi",            
    16: "Unknown_16", 
    17: "Kankrej",            
    18: "Malnad",             
    19: "Kherigarh",          
    20: "Unknown_20", 
    21: "Khillar",            
    22: "Dagri",              
    23: "Badri",              
    24: "Unknown_24", 
    25: "Unknown_25", 
    26: "Bachaur",            
    27: "Chilika",            
    28: "Siri",               
    29: "Nimari",             
    30: "Unknown_30", 
    31: "Kangayam",           
    32: "Khariar",            
    33: "Unknown_33", 
    34: "Poda Thurpu",        
    35: "Sahiwal",            
    36: "Unknown_36", 
    37: "Unknown_37", 
    38: "Murrah Buffalo",     
    39: "Umblachery",         
    40: "Motu"                
}

# -----------------------------
# Load EfficientNet classifier
# -----------------------------
classifier = models.efficientnet_b0(weights=None)
classifier.classifier[1] = torch.nn.Linear(
    classifier.classifier[1].in_features, num_classes
)

# Load trained weights safely
state_dict = torch.load("efficientnet_70.pth", map_location=device)
classifier.load_state_dict(state_dict, strict=False)  # strict=False handles mismatch
classifier = classifier.to(device).eval()

# -----------------------------
# Load YOLOv8 detection model
# -----------------------------
if os.path.exists("yolov8n.pt"):
    detector = YOLO("yolov8n.pt")  # pretrained COCO model
else:
    print('YOLO model missing')

# -----------------------------
# Preprocessing for classifier
# -----------------------------
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.5, 0.5, 0.5], std=[0.5, 0.5, 0.5])
])

# -----------------------------
# Detection + Classification
# -----------------------------
def detect_and_classify(image_path, conf_threshold=0.3):
    # Load image
    image = Image.open(image_path).convert("RGB")

    # YOLO detection (with confidence threshold)
    results = detector(image_path, imgsz=640, conf=conf_threshold, verbose=False)[0]

    cow_detected = False

    for box, cls, conf in zip(results.boxes.xyxy, results.boxes.cls, results.boxes.conf):
        cls_id = int(cls)
        conf = float(conf)
        
        # Check for cow (COCO: class 19 = cow)
        if cls_id == 19 and conf >= conf_threshold:
            cow_detected = True
            x1, y1, x2, y2 = map(int, box)
            cropped = image.crop((x1, y1, x2, y2))

            # Preprocess for classifier
            input_tensor = transform(cropped).unsqueeze(0).to(device)

            # Classification
            with torch.no_grad():
                output = classifier(input_tensor)
                pred = torch.argmax(output, 1)
                class_idx = pred.item()
                breed_name = class_names[class_idx]

            print(f"✅ Cattle detected! Breed: {breed_name} (Class {class_idx}) (YOLO conf {conf:.2f})")

    if not cow_detected:
        print("⚠ No cow detected.")

In [29]:
# -----------------------------
# Test folder images
# -----------------------------
# test_folder = r"D:\model\images"
# for img_file in os.listdir(test_folder):
#     if img_file.lower().endswith(".jpg") or img_file.lower().endswith(".png"):
#         img_path = os.path.join(test_folder, img_file)
#         print(f"\n Processing {img_file}...")
#         detect_and_classify(img_path)
detect_and_classify("D:\model\images\holstein.jpg")

✅ Cattle detected! Breed: Holstein (Class 13) (YOLO conf 0.74)


<>:10: SyntaxWarning: invalid escape sequence '\m'
<>:10: SyntaxWarning: invalid escape sequence '\m'
C:\Users\ganap\AppData\Local\Temp\ipykernel_36516\711460985.py:10: SyntaxWarning: invalid escape sequence '\m'
  detect_and_classify("D:\model\images\holstein.jpg")
